In [1]:
import sys
sys.path.append("..")
import torch
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm
import gc
from datasets import load_from_disk

from src.data import PROJECT_ROOT
from src.llm_upgrade import wrap_for_transformer_lens
from src.dataset_cot import prepare_direct_pythia_dataset
from src.sae import TopKSAE, collect_all_activations, create_activation_dataloader, save_activations, load_activations

# Параметры построения SAE

In [2]:
EXP_ID = "exp3-2"
MODEL_NAME = "EleutherAI/pythia-1b-deduped"
VARIANT = "depth-0"
ADAPTER_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-6000")

In [3]:
PROBING_JSON = PROJECT_ROOT / "results/probing/probe_1b(qLoRA)_depth-0_resid_post_last.json"
with open(PROBING_JSON, "r", encoding="utf-8") as f:
    probing_data = json.load(f)
BEST_LAYER = probing_data["summary"]["best_layer"]
print(f"Загружен слой: {BEST_LAYER} (probing acc: {probing_data['summary']['best_dev_acc']:.4f})")

Загружен слой: 13 (probing acc: 0.8528)


In [4]:
HOOK_POINT = f"blocks.{BEST_LAYER}.hook_resid_post"

In [ ]:
D_SAE = 8192
K = 32              # жёсткая разреженность (Top-K)
LR = 3e-4           # скорость обучения
BATCH_SIZE = 128    # размер батча активаций
MAX_LENGTH = 512

In [6]:
SAVE_DIR = str(PROJECT_ROOT / f"results/sae/{EXP_ID}/layer_{BEST_LAYER}")
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

# Загрузка модели и датасета

In [7]:
# Загрузка модели и данных
hooked_model, tokenizer = wrap_for_transformer_lens(MODEL_NAME, ADAPTER_PATH, device="cpu")
hooked_model.eval()
hooked_model.to("cuda")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer
Moving model to device:  cuda


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-15): 16 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
    

In [8]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
raw_dataset = load_from_disk(str(cache_path))
train_dataset = prepare_direct_pythia_dataset(raw_dataset["train"])

In [9]:
d_model = hooked_model.cfg.d_model

In [10]:
print(f"Датасет: {len(train_dataset)} примеров | d_model={d_model}")

Датасет: 60557 примеров | d_model=2048


# Сбор активаций

In [12]:
# Путь для сохранения активаций
ACTIVATIONS_PATH = PROJECT_ROOT / f"results/sae/{EXP_ID}/layer_{BEST_LAYER}" / "activations.pt"

In [13]:
all_activations = collect_all_activations(
    dataset=train_dataset,
    model=hooked_model,
    tokenizer=tokenizer,
    hook_point=HOOK_POINT,
    max_length=MAX_LENGTH,
    collect_batch_size=4,  # Маленький батч для сбора
    prepend_bos=True,
    device="cuda"
)

Сбор активаций: 60557 примеров, batch_size=4


Collecting: 100%|██████████| 15140/15140 [1:31:27<00:00,  2.76it/s]


Активации собраны: torch.Size([60557, 2048]) | Память: 473.1 MB


In [14]:
metadata = {
    "model": MODEL_NAME,
    "variant": VARIANT,
    "layer": BEST_LAYER,
    "hook_point": HOOK_POINT,
    "max_length": MAX_LENGTH,
    "n_samples": len(all_activations)
}
save_activations(all_activations, ACTIVATIONS_PATH, metadata=metadata)

Активации сохранены: C:\MyPythonProjects\mephi_diss\results\sae\exp3-2\layer_13\activations.pt | Размер: 473.1 MB


WindowsPath('C:/MyPythonProjects/mephi_diss/results/sae/exp3-2/layer_13/activations.pt')

In [15]:
# Сбор статистик для диагностики
mean = all_activations.mean(dim=0, keepdim=True)
scale = (all_activations - mean).norm(dim=1).mean()
print(f"Статистики: mean.shape={mean.shape}, scale={scale:.4f}")

Статистики: mean.shape=torch.Size([1, 2048]), scale=40.0505


In [16]:
# Создание DataLoader для обучения
train_loader = create_activation_dataloader(all_activations,
                                            batch_size=BATCH_SIZE,
                                            shuffle=True)
print(f"{len(train_loader)} батчей")

474 батчей


# SAE

In [17]:
# Очистка памяти перед запуском
torch.cuda.empty_cache()
gc.collect()

35

In [18]:
print(f"Инициализация SAE: d_in={d_model}, d_sae={D_SAE}, k={K}, batch_size={BATCH_SIZE}")

Инициализация SAE: d_in=2048, d_sae=8192, k=32, batch_size=128


In [19]:
# Инициализация модели и оптимизатора
sae = TopKSAE(d_in=d_model, d_sae=D_SAE, k=K).to("cuda")
optimizer = torch.optim.Adam(sae.parameters(), lr=1e-4, betas=(0.9, 0.999))

In [20]:
# Параметры обучения
NUM_EPOCHS = 3
total_steps = len(train_loader) * NUM_EPOCHS
log_interval, save_interval = 200, 500
loss_history = []
global_step = 0

In [21]:
print(f"Всего шагов: {total_steps}")

Всего шагов: 1422


In [22]:
for epoch in range(NUM_EPOCHS):
    for batch_acts, in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        batch_acts = batch_acts.to("cuda")  # Быстрая передача благодаря pin_memory

        optimizer.zero_grad()
        sparse, recon = sae(batch_acts)
        loss = (recon - batch_acts).pow(2).sum(dim=-1).mean()
        loss.backward()

        # Нормализация декодера после каждого шага
        with torch.no_grad():
            sae.W_dec.data /= sae.W_dec.data.norm(dim=0, keepdim=True)
        optimizer.step()

        loss_history.append(loss.item())
        if global_step % log_interval == 0:
            print(f"Step {global_step:5d} | Loss: {loss.item():.4f}")
        if global_step % save_interval == 0 and global_step > 0:
            torch.save({
                "step": global_step,
                "model_state_dict": sae.state_dict(),
                "loss_history": loss_history
            }, f"{SAVE_DIR}/checkpoint_{global_step}.pt")
        global_step += 1

Epoch 1/3:   2%|▏         | 8/474 [00:00<00:23, 19.89it/s]

Step     0 | Loss: 7164.7246


Epoch 1/3:  44%|████▎     | 207/474 [00:03<00:03, 71.88it/s]

Step   200 | Loss: 578.7512


Epoch 1/3:  86%|████████▌ | 407/474 [00:06<00:00, 70.92it/s]

...

Epoch 3/3:  56%|█████▌    | 264/474 [00:03<00:02, 71.22it/s]

Step  1200 | Loss: 303.8754


Epoch 3/3:  97%|█████████▋| 462/474 [00:06<00:00, 70.72it/s]

Step  1400 | Loss: 282.6326


Epoch 3/3: 100%|██████████| 474/474 [00:06<00:00, 68.77it/s]


In [23]:
# Сохранение финального шага
torch.save({
    "step": global_step,
    "model_state_dict": sae.state_dict(),
    "loss_history": loss_history,
    "config": {
        "d_in": d_model,
        "d_sae": D_SAE,
        "k": K,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE
    }
}, f"{SAVE_DIR}/sae_final.pt")

# Диагностика качества SAE

In [24]:
# Загрузка финальной модели
checkpoint = torch.load(f"{SAVE_DIR}/sae_final.pt", map_location="cuda", weights_only=False)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()
mean_gpu = mean.to("cuda")
scale_gpu = scale.to("cuda")

In [25]:
# Тестовый батч для оценки
test_texts = [train_dataset[i]["text"] for i in range(128)]
tokens = hooked_model.to_tokens(test_texts, prepend_bos=True).to("cuda")
if MAX_LENGTH and tokens.shape[1] > MAX_LENGTH:
    tokens = tokens[:, :MAX_LENGTH]

In [26]:
with torch.no_grad():
    _, cache = hooked_model.run_with_cache(tokens, names_filter=lambda n: n == HOOK_POINT)
    acts = cache[HOOK_POINT]
    lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in test_texts]
    batch_acts = torch.stack([acts[i, min(lengths[i], acts.shape[1]-1), :] for i in range(len(test_texts))]).to(torch.float32)

In [27]:
# Прогон через SAE
sparse, recon = sae(batch_acts)

In [28]:
# Метрики
explained_var = 1 - ((batch_acts - recon).pow(2).sum() / batch_acts.pow(2).sum()).item()
sparsity = (sparse != 0).float().mean().item()
target_sparsity = K / D_SAE

In [29]:
print(f"Explained Variance: {explained_var:.4f}")
print(f"Фактическая разреженность: {sparsity:.4f}")
print(f"Целевая разреженность (K/D_SAE): {target_sparsity:.4f}")

Explained Variance: 0.9623
Фактическая разреженность: 0.0039
Целевая разреженность (K/D_SAE): 0.0039
